# Data Cleaning & Preprocessing Pipeline

This Jupyter Notebook (`.ipynb`) automates data ingestion, cleaning, feature engineering, and relational integrity validation using `pandas` to prepare datasets for SQL database import.

---

## Key Pipeline Steps

### 1. Data Ingestion
* Load multiple raw source CSV files into separate `pandas` DataFrames.

### 2. Initial Data Cleaning
* Drop duplicate entries, handle null/missing values, and strip unwanted whitespace.
* Remove irrelevant columns and filter out invalid rows.

### 3. Data Merging & Integration
* Perform `JOIN` / `merge` operations across primary datasets on common keys.
* Standardize schema formats and data types (e.g., datetimes, numeric values).

### 4. Feature Engineering
* Create derived metrics, categorical flags, and aggregated indicators.
* Export key transformed subsets into new standalone processed CSV files.

### 5. Relational & Foreign Key Validation
* Validate key integrity (`company_id`, `job_id`, etc.) across foreign key relationships.
* Strip orphan rows (records with references missing in primary key tables) to enforce strict referential integrity prior to SQL loading.
* Export final, cleaned datasets ready for relational database ingestion.

### Loading Data

### 1. Setup & Data Loading
Import the required libraries, configure dataset paths, and load all raw source datasets before beginning the cleaning workflow.


In [ ]:
# importing necessary libraries

import os
import pandas as pd
import matplotlib.pyplot as plt

# setting up the paths for the datasets
notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath("..")

In [ ]:
# loading the datasets using specified paths
companies_df_org = pd.read_csv(os.path.join(parent_dir, "raw_datasets", "companies.csv"))
company_ind_df = pd.read_csv(os.path.join(parent_dir, "raw_datasets", "company_industries.csv"))
company_spec_df = pd.read_csv(os.path.join(parent_dir, "raw_datasets", "company_specialities.csv"))
employee_counts_df = pd.read_csv(os.path.join(parent_dir, "raw_datasets", "employee_counts.csv"))

In [ ]:
# loading the job postings dataset
org_dataset_path = os.path.join(parent_dir, "raw_datasets", "job_postings.csv")
job_postings_df = pd.read_csv(org_dataset_path)

In [ ]:
# loading the job skills and job industries datasets
df1 = pd.read_csv(os.path.join(parent_dir, "raw_datasets", "job_skills.csv"))
df2 = pd.read_csv(os.path.join(parent_dir, "raw_datasets", "job_industries.csv"))

### 2. Companies & Company Details
Prepare the companies data, inspect related company tables, build the company details dataset, standardize its recorded time, validate company relationships, and export the cleaned files.


#### Making file#1 & #2 (companies & company_details)

#### 2.1 Clean the Companies Dataset
Keep the required company columns and inspect the location fields for unusually long values.


In [ ]:
# cleaning the companies dataset to keep only the required columns
companies_df = companies_df_org[['company_id', 'company_name', 'city', 'state', 'country', 'url']]

In [ ]:
# removing rows with city names longer than 100 characters
companies_df.drop(companies_df[companies_df.city.str.len() > 100].index, inplace=True)

In [ ]:
# removing rows with state names longer than 99 characters
companies_df[companies_df.state.str.len() > 99]

,company_id,company_name,city,state,country,url


In [ ]:
# removing rows with state names longer than 99 characters
companies_df[companies_df.country.str.len() > 99]

,company_id,company_name,city,state,country,url


In [1]:
# standardizing states by mapping out the duplicates, misspells, and variations in the state names
state_mapping = {
    # United States
    "AL": "Alabama",
    "Al": "Alabama",
    "Alabama": "Alabama",
    
    "AK": "Alaska",
    "Alaska": "Alaska",
    
    "AZ": "Arizona",
    "Az": "Arizona",
    "Arizona": "Arizona",
    
    "AR": "Arkansas",
    "Arkansas": "Arkansas",
    
    "CA": "California",
    "ca": "California",
    "Ca": "California",
    "CA.": "California",
    "CA - California": "California",
    "Calif.": "California",
    "California": "California",
    "CALIFORNIA": "California",
    "California ": "California",
    
    "CO": "Colorado",
    "Co": "Colorado",
    "co": "Colorado",
    "CO.": "Colorado",
    "Colorado": "Colorado",
    
    "CT": "Connecticut",
    "ct": "Connecticut",
    "CT ": "Connecticut",
    "Connecticut": "Connecticut",
    
    "DE": "Delaware",
    "Delaware": "Delaware",
    "DELAWARE (DE)": "Delaware",
    
    "DC": "District of Columbia",
    "D.C.": "District of Columbia",
    "District of Columbia": "District of Columbia",
    "Washington D.C.": "District of Columbia",
    
    "FL": "Florida",
    "fl": "Florida",
    "Fl": "Florida",
    "FL ": "Florida",
    "Florida": "Florida",
    "FLORIDA": "Florida",
    
    "GA": "Georgia",
    "ga": "Georgia",
    "GA ": "Georgia",
    "Georgia": "Georgia",
    "GEORGIA": "Georgia",
    "Georgia ": "Georgia",
    "Georgia (GA)": "Georgia",
    
    "HI": "Hawaii",
    "Hawaii": "Hawaii",
    
    "ID": "Idaho",
    "Idaho": "Idaho",
    
    "IL": "Illinois",
    "il": "Illinois",
    "Il": "Illinois",
    "Illinois": "Illinois",
    "Illinois ": "Illinois",
    "Ilinois": "Illinois",
    "Illinos": "Illinois",
    "Lincolnwood, Illinois": "Illinois",
    
    "IN": "Indiana",
    "In": "Indiana",
    "Indiana": "Indiana",
    
    "IA": "Iowa",
    "iowa": "Iowa",
    "Iowa": "Iowa",
    
    "KS": "Kansas",
    "Kansas": "Kansas",
    
    "KY": "Kentucky",
    "Kentucky": "Kentucky",
    
    "LA": "Louisiana",
    "Louisiana": "Louisiana",
    "Louisiana ": "Louisiana",
    
    "ME": "Maine",
    "Maine": "Maine",
    
    "MD": "Maryland",
    "md": "Maryland",
    "Maryland": "Maryland",
    "maryland": "Maryland",
    "MARYLAND": "Maryland",
    
    "MA": "Massachusetts",
    "ma": "Massachusetts",
    "Ma": "Massachusetts",
    "MA ": "Massachusetts",
    "Massachusetts": "Massachusetts",
    " MA ": "Massachusetts",
    
    "MI": "Michigan",
    "mi": "Michigan",
    "Mi": "Michigan",
    "Michigan": "Michigan",
    "MICHIGAN": "Michigan",
    
    "MN": "Minnesota",
    "Minnesota": "Minnesota",
    "Minnesota ": "Minnesota",
    
    "MS": "Mississippi",
    "Mississippi": "Mississippi",
    
    "MO": "Missouri",
    "Missouri": "Missouri",
    
    "MT": "Montana",
    "Montana": "Montana",
    
    "NE": "Nebraska",
    "Nebraska": "Nebraska",
    
    "NV": "Nevada",
    "Nevada": "Nevada",
    
    "NH": "New Hampshire",
    "New Hampshire": "New Hampshire",
    
    "NJ": "New Jersey",
    "N.J.": "New Jersey",
    "New Jersey": "New Jersey",
    "New Jersey ": "New Jersey",
    "NewJersey": "New Jersey",
    
    "NM": "New Mexico",
    "New Mexico": "New Mexico",
    
    "NY": "New York",
    "NY ": "New York",
    "NY - New York": "New York",
    "New York": "New York",
    "New York ": "New York",
    "NewYork": "New York",
    
    "NC": "North Carolina",
    "North Carolina": "North Carolina",
    "North Carolina ": "North Carolina",
    
    "ND": "North Dakota",
    "North Dakota": "North Dakota",
    
    "OH": "Ohio",
    "Ohio": "Ohio",
    
    "OK": "Oklahoma",
    "Oklahoma": "Oklahoma",
    
    "OR": "Oregon",
    "Oregon": "Oregon",
    
    "PA": "Pennsylvania",
    "pa": "Pennsylvania",
    "Pa": "Pennsylvania",
    "PA ": "Pennsylvania",
    "Pennsylvania": "Pennsylvania",
    "Pennsylvania ": "Pennsylvania",
    
    "RI": "Rhode Island",
    "Rhode Island": "Rhode Island",
    
    "SC": "South Carolina",
    "South Carolina": "South Carolina",
    
    "SD": "South Dakota",
    "South Dakota": "South Dakota",
    
    "TN": "Tennessee",
    "Tennessee": "Tennessee",
    "TENNESSEE": "Tennessee",
    
    "TX": "Texas",
    "tx": "Texas",
    "Tx": "Texas",
    "TX ": "Texas",
    "Texas": "Texas",
    "TEXAS": "Texas",
    "Texas ": "Texas",
    
    "UT": "Utah",
    "Utah": "Utah",
    
    "VT": "Vermont",
    "Vermont": "Vermont",
    
    "VA": "Virginia",
    "VA ": "Virginia",
    "Virginia": "Virginia",
    
    "WA": "Washington",
    "wa": "Washington",
    "Wa": "Washington",
    "WA ": "Washington",
    "Washington": "Washington",
    "Washington ": "Washington",
    
    "WV": "West Virginia",
    
    "WI": "Wisconsin",
    "wi": "Wisconsin",
    "Wi": "Wisconsin",
    "Wisconsin": "Wisconsin",
    
    "WY": "Wyoming",
    "WY ": "Wyoming",
    "Wyoming": "Wyoming",

    # Canada
    "AB": "Alberta",
    "Alberta": "Alberta",
    "BC": "British Columbia",
    "British Columbia": "British Columbia",
    "MB": "Manitoba",
    "Manitoba": "Manitoba",
    "ON": "Ontario",
    "Ontario": "Ontario",
    "ONTARIO": "Ontario",
    "QC": "Quebec",
    "Quebec": "Quebec",
    "Québec": "Quebec",
    "SK": "Saskatchewan",
    "Saskatchewan": "Saskatchewan",

    # United Kingdom
    "England": "England",
    "ENGLAND": "England",
    "Scotland": "Scotland",
    "Greater London": "Greater London",
    "Central London ": "Greater London",
    "Bedfordshire": "Bedfordshire",
    "Berkshire": "Berkshire",
    "Buckinghamshire": "Buckinghamshire",
    "Bucks": "Buckinghamshire",
    "Cambridgeshire": "Cambridgeshire",
    "Cambs": "Cambridgeshire",
    "Cheshire": "Cheshire",
    "East Sussex": "East Sussex",
    "Gloucestershire": "Gloucestershire",
    "Hampshire": "Hampshire",
    "Hertfordshire": "Hertfordshire",
    "Herts": "Hertfordshire",
    "Middlesex": "Middlesex",
    "Surrey": "Surrey",
    "Sussex": "Sussex",
    "West Midlands": "West Midlands",
    "London": "Greater London",
    "Luton": "Luton",
    "Weybridge": "Weybridge",
    "Strand": "Strand",
    "Antrim": "Antrim",

    # Europe
    "Baden-Württemberg": "Baden-Württemberg",
    "Bavaria": "Bavaria",
    "Bayern": "Bavaria",
    "Hessen": "Hesse",
    "Nordrhein-Westfalen": "North Rhine-Westphalia",
    "North-Rhine-Westphalia": "North Rhine-Westphalia",
    "North Rhine-Westphalia": "North Rhine-Westphalia",
    "North Rhine-Westphalia ": "North Rhine-Westphalia",
    "NRW": "North Rhine-Westphalia",
    
    "Ile-de-France": "Île-de-France",
    "Île-de-France": "Île-de-France",
    "Ile de France": "Île-de-France",
    "Occitanie": "Occitanie",
    "Auvergne-Rhône-Alpes": "Auvergne-Rhône-Alpes",
    "Yvelines": "Yvelines",
    
    "Madrid": "Community of Madrid",
    "Community of Madrid": "Community of Madrid",
    "Barcelona": "Catalonia",
    "Córdoba": "Córdoba",
    "Spain": "Spain",
    
    "Italia": "Italy",
    "Italy": "Italy",
    "Florence": "Tuscany",
    "Parma": "Emilia-Romagna",
    
    "Switzerland": "Switzerland",
    "Basel-Country": "Basel-Landschaft",
    "Baselstadt": "Basel-Stadt",
    "Geneva": "Geneva",
    "St. Gallen": "St. Gallen",
    "Zug": "Zug",
    "ZG": "Zug",
    "Zurich": "Zurich",
    "Zürich": "Zurich",
    
    "Sweden": "Sweden",
    "Stockholm": "Stockholm",
    "Stockholm County": "Stockholm County",
    "Skane": "Skåne",
    "Skane County": "Skåne",
    
    "Finland": "Finland",
    "Helsinki": "Uusimaa",
    "Uusimaa": "Uusimaa",
    
    "Iceland": "Iceland",
    "Ireland": "Ireland",
    "Dublin": "Dublin",
    "Prague": "Prague",
    "Rogaland": "Rogaland",
    "Styria": "Styria",
    "Walloon Region": "Walloon Region",
    "Utrecht": "Utrecht",
    "Noord-Holland": "North Holland",
    "North Holland": "North Holland",
    "Zuid-Holland": "South Holland",

    # Asia / Middle East
    "Delhi": "Delhi",
    "Gujarat": "Gujarat",
    "Haryana": "Haryana",
    "Harayana ": "Haryana",
    "Karnataka": "Karnataka",
    "KARNATAKA": "Karnataka",
    "Kerala": "Kerala",
    "Maharashtra": "Maharashtra",
    "Maharastra": "Maharashtra",
    "Tamil Nadu": "Tamil Nadu",
    "Tamilnadu": "Tamil Nadu",
    "Telangana": "Telangana",
    "Uttar Pradesh": "Uttar Pradesh",
    "UP": "Uttar Pradesh",
    
    "Gyeonggi-do": "Gyeonggi-do",
    "Seoul": "Seoul",
    "Jiangsu": "Jiangsu",
    "Changzhou": "Jiangsu",
    "Huangpu": "Shanghai",
    "Shanghai": "Shanghai",
    "Guangdong": "Guangdong",
    "广东省": "Guangdong",
    "New Taipei City": "New Taipei City",
    "Taiwan": "Taiwan",
    "Osaka": "Osaka",
    "Tokyo": "Tokyo",
    "東京都": "Tokyo",
    "Kanagawa ": "Kanagawa",
    "Hong Kong": "Hong Kong",
    "Singapore": "Singapore",
    "Dubai": "Dubai",
    "Abu Dhabi": "Abu Dhabi",
    
    # Australia / New Zealand
    "NSW": "New South Wales",
    "New South Wales": "New South Wales",
    "QLD": "Queensland",
    "Queensland": "Queensland",
    "VIC": "Victoria",
    "Victoria": "Victoria",
    "Western Australia": "Western Australia",
    "Bay of Plenty": "Bay of Plenty",
    "Auckland": "Auckland",

    # Latin America
    "buenos aires": "Buenos Aires",
    "Jalisco": "Jalisco",
    "Nuevo León": "Nuevo León",
    "treinta y tres": "Treinta y Tres",

    # Other
    "Bermuda": "Bermuda",
    "Israel": "Israel",
    "Hrvatska": "Croatia",
    "China": "China",
    "France": "France",
    "Germany": "Germany",
    "USA": "United States",
    "United States": "United States",

    # Invalid / non-state values
    "0": "Unknown",
    "1824": "Unknown",
    "28801": "Unknown",
    "94086": "Unknown",
    "Anywhere": "Unknown",
}

companies_df["state"] = companies_df["state"].replace(state_mapping)

NameError: name 'companies_df' is not defined

#### 2.2 Inspect Related Company Data
Review the related company datasets before removing duplicate company records and combining them into the company details dataset.


In [ ]:
# displaying the information of the dataframes to check for any inconsistencies or issues
company_ind_df.info(), company_spec_df.info(), employee_counts_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15880 entries, 0 to 15879
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   company_id  15880 non-null  int64
 1   industry    15880 non-null  str  
dtypes: int64(1), str(1)
memory usage: 248.3 KB
<class 'pandas.DataFrame'>
RangeIndex: 128355 entries, 0 to 128354
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   company_id  128355 non-null  int64
 1   speciality  128355 non-null  str  
dtypes: int64(1), str(1)
memory usage: 2.0 MB
<class 'pandas.DataFrame'>
RangeIndex: 15907 entries, 0 to 15906
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   company_id      15907 non-null  int64  
 1   employee_count  15907 non-null  int64  
 2   follower_count  15907 non-null  int64  
 3   time_recorded   15907 non-null  float64
dtypes: float64(1), int64(3

(None, None, None)

#### 2.3 Build Company Details
Remove duplicate company records from the supporting tables and merge the company information into a single company details dataframe.


In [ ]:
# removing duplicate rows based on 'company_id' column, keeping the first occurrence
company_ind_df = company_ind_df.drop_duplicates(subset='company_id', keep='first')
company_spec_df = company_spec_df.drop_duplicates(subset='company_id', keep='first')
employee_counts_df = employee_counts_df.drop_duplicates(subset='company_id', keep='first')

# merging the dataframes to create a comprehensive company details dataframe
company_details_df_beta = pd.merge(company_ind_df, company_spec_df, on='company_id', how='inner').merge(employee_counts_df, on='company_id', how='inner')
company_details_df = pd.merge(companies_df_org[['company_id', 'company_size']], company_details_df_beta, on='company_id', how='inner')

company_details_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4452 entries, 0 to 4451
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   company_id      4452 non-null   int64  
 1   company_size    4146 non-null   float64
 2   industry        4452 non-null   str    
 3   speciality      4452 non-null   str    
 4   employee_count  4452 non-null   int64  
 5   follower_count  4452 non-null   int64  
 6   time_recorded   4452 non-null   float64
dtypes: float64(2), int64(3), str(2)
memory usage: 243.6 KB


fixing time

#### 2.4 Standardize Time & Validate Relationships
Convert the recorded time to datetime format and retain only company details associated with companies in the cleaned companies dataset.


In [ ]:
# converting the 'time_recorded' column to datetime format for better readability and analysis
company_details_df['time_recorded'] = pd.to_datetime(company_details_df['time_recorded'], unit='s')

In [ ]:
# filtering the company_details_df to keep only the rows with company_id present in companies_df
company_details_df = company_details_df[company_details_df['company_id'].isin(companies_df['company_id'])]

,company_id,company_size,industry,speciality,employee_count,follower_count,time_recorded
0,1009,7.0,Information Technology & Services,Cloud,316130,16114398,2023-08-24 04:19:39
1,1016,7.0,Hospital & Health Care,Healthcare,53495,2060378,2023-08-24 04:55:10
2,1021,7.0,Renewables & Environment,Distributed Power,26963,2340835,2023-08-24 08:38:46
3,1028,7.0,Information Technology & Services,enterprise,202019,9289332,2023-08-24 07:12:12
4,1038,7.0,Management Consulting,Audit,437314,13423472,2023-08-24 03:37:45
...,...,...,...,...,...,...,...
4447,98383042,NaN,Staffing & Recruiting,Radiation Oncology,2,20,2023-08-23 23:00:05
4448,98470447,NaN,Higher Education,STEM,1,42,2023-08-24 06:34:17
4449,98488356,1.0,Staffing & Recruiting,Staff Augmentation,3,128,2023-08-24 00:09:44
4450,98525576,1.0,Shipbuilding,Scaffolding,1,72,2023-08-24 06:36:19


#### 2.5 Export Company Datasets
Save the cleaned companies and company details datasets for downstream relational analysis.


In [ ]:
# saving the cleaned datasets to the 'cleaned_datasets' directory for further analysis or processing
companies_df.to_csv(os.path.join(parent_dir, "cleaned_datasets", "3_companies.csv"), index=False)
company_details_df.to_csv(os.path.join(parent_dir, "cleaned_datasets", "4_company_details.csv"), index=False)

### 3. Job Postings
Inspect the raw job postings data, keep the required fields, review its structure and numerical values, standardize selected data types, validate company references, and export the cleaned dataset.


### cleaning file#3 (job_postings dataset)

#### 3.1 Inspect Job Postings
Review representative rows, dataframe information, unique values, and missing-value counts before transforming the dataset.


In [ ]:
# displaying a random sample of 3 rows
job_postings_df.sample(3)

,job_id,company_id,title,max_salary,med_salary,min_salary,pay_period,work_type,location,applies,...,expiry,closed_time,experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type.1,currency,compensation_type
7594,3697340789,90639.0,Site Civil Engineer,120000.0,NaN,100000.0,YEARLY,Full-time,"New York, NY",53.0,...,1.700000e+12,NaN,Mid-Senior level,NaN,1.690000e+12,NaN,1,FULL_TIME,USD,BASE_SALARY
10652,3699081042,30086.0,"Director, Systems Engineering - Major Accounts",NaN,NaN,NaN,NaN,Full-time,"Miami, FL",1.0,...,1.700000e+12,NaN,Mid-Senior level,NaN,1.690000e+12,jobs.smartrecruiters.com,0,FULL_TIME,NaN,NaN
4019,3693071158,69094979.0,Accountant,72000.0,NaN,60000.0,YEARLY,Full-time,"Menomonee Falls, WI",NaN,...,1.700000e+12,1.690000e+12,Mid-Senior level,NaN,1.690000e+12,NaN,1,FULL_TIME,USD,BASE_SALARY


In [ ]:
# displaying the information of the job_postings_df to check for any inconsistencies or issues
job_postings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15886 entries, 0 to 15885
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   job_id                15886 non-null  int64  
 1   company_id            15520 non-null  float64
 2   title                 15886 non-null  str    
 3   max_salary            5521 non-null   float64
 4   med_salary            981 non-null    float64
 5   min_salary            5521 non-null   float64
 6   pay_period            6502 non-null   str    
 7   work_type             15886 non-null  str    
 8   location              15886 non-null  str    
 9   applies               8700 non-null   float64
 10  original_listed_time  15886 non-null  float64
 11  remote_allowed        2340 non-null   float64
 12  views                 13123 non-null  float64
 13  job_posting_url       15886 non-null  str    
 14  application_url       9795 non-null   str    
 15  application_type      15886 no

In [ ]:
# checking for the number of unique values in each column of the job_postings_df
job_postings_df.nunique()

job_id                  15886
company_id               6030
title                   10956
max_salary               1314
med_salary                314
min_salary               1233
pay_period                  3
work_type                   7
location                 3010
applies                   285
original_listed_time        1
remote_allowed              1
views                     713
job_posting_url         15886
application_url          8899
application_type            3
expiry                      3
closed_time                 1
experience_level            6
skills_desc               141
listed_time                 1
posting_domain           1296
sponsored                   2
work_type.1                 7
currency                    1
compensation_type           1
dtype: int64

In [ ]:
# checking for the number of missing values in each column of the job_postings_df
job_postings_df.isna().sum()

job_id                      0
company_id                366
title                       0
max_salary              10365
med_salary              14905
min_salary              10365
pay_period               9384
work_type                   0
location                    0
applies                  7186
original_listed_time        0
remote_allowed          13546
views                    2763
job_posting_url             0
application_url          6091
application_type            0
expiry                      0
closed_time             14958
experience_level         4902
skills_desc             15742
listed_time                 0
posting_domain           6842
sponsored                   0
work_type.1                 0
currency                 9384
compensation_type        9384
dtype: int64

#### 3.2 Select and Inspect Job Posting Fields
Keep the required columns, then inspect the resulting structure and numerical columns for consistency and unusual values.


In [ ]:
# cleaning the job_postings_df to keep only the required columns
job_postings_df = job_postings_df[
        ['job_id', 
         'company_id', 
         'title', 
         'max_salary', 
         'min_salary', 
         'work_type', 
         'location', 
         'applies', 
         'remote_allowed', 
         'views', 
         'job_posting_url', 
         'application_url', 
         'application_type', 
         'experience_level', 
         'sponsored']]
job_postings_df.head(3)

,job_id,company_id,title,max_salary,min_salary,work_type,location,applies,remote_allowed,views,job_posting_url,application_url,application_type,experience_level,sponsored
0,85008768,NaN,Licensed Insurance Agent,52000.0,45760.0,Full-time,"Chico, CA",NaN,NaN,5.0,https://www.linkedin.com/jobs/view/85008768/?t...,NaN,ComplexOnsiteApply,NaN,1
1,133114754,77766802.0,Sales Manager,NaN,NaN,Full-time,"Santa Clarita, CA",NaN,NaN,NaN,https://www.linkedin.com/jobs/view/133114754/?...,NaN,ComplexOnsiteApply,NaN,0
2,133196985,1089558.0,Model Risk Auditor,NaN,NaN,Contract,"New York, NY",1.0,NaN,17.0,https://www.linkedin.com/jobs/view/133196985/?...,NaN,ComplexOnsiteApply,NaN,0
3,381055942,96654609.0,Business Manager,NaN,NaN,Full-time,"Forney, TX",NaN,NaN,NaN,https://www.linkedin.com/jobs/view/381055942/?...,NaN,ComplexOnsiteApply,NaN,0
4,529257371,1244539.0,NY Studio Assistant,NaN,NaN,Full-time,"New York, NY",NaN,NaN,2.0,https://www.linkedin.com/jobs/view/529257371/?...,NaN,ComplexOnsiteApply,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15881,3701373516,74718032.0,Sanitation Technician,NaN,NaN,Part-time,"West Columbia, SC",NaN,NaN,1.0,https://www.linkedin.com/jobs/view/3701373516/...,https://aspirebakeriescareers.com/job/West-Col...,OffsiteApply,Entry level,0
15882,3701373522,38897.0,Unit Secretary,NaN,NaN,Full-time,"Teaneck, NJ",2.0,NaN,7.0,https://www.linkedin.com/jobs/view/3701373522/...,https://recruiting.ultipro.com/HOL1005HNMC/Job...,OffsiteApply,Entry level,0
15883,3701373523,38897.0,"Radiology Aide, Perdiem",NaN,NaN,Part-time,"Teaneck, NJ",NaN,NaN,3.0,https://www.linkedin.com/jobs/view/3701373523/...,https://recruiting.ultipro.com/HOL1005HNMC/Job...,OffsiteApply,Entry level,0
15884,3701373524,2623.0,MRI Manager,135000.0,110000.0,Full-time,"New York, NY",NaN,NaN,10.0,https://www.linkedin.com/jobs/view/3701373524/...,https://opportunities.columbia.edu/jobs/mri-ma...,OffsiteApply,Mid-Senior level,0


In [ ]:
# checking all columns' value counts and dtypes
job_postings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15886 entries, 0 to 15885
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   job_id            15886 non-null  int64  
 1   company_id        15520 non-null  float64
 2   title             15886 non-null  str    
 3   max_salary        5521 non-null   float64
 4   min_salary        5521 non-null   float64
 5   work_type         15886 non-null  str    
 6   location          15886 non-null  str    
 7   applies           8700 non-null   float64
 8   remote_allowed    2340 non-null   float64
 9   views             13123 non-null  float64
 10  job_posting_url   15886 non-null  str    
 11  application_url   9795 non-null   str    
 12  application_type  15886 non-null  str    
 13  experience_level  10984 non-null  str    
 14  sponsored         15886 non-null  int64  
dtypes: float64(6), int64(2), str(7)
memory usage: 1.8 MB


In [ ]:
# checking for any outliers in the numerical columns of the job_postings_df using descriptive statistics
job_postings_df[['max_salary', 'min_salary', 'applies', 'views']].agg(['mean', 'median', 'std', 'min', 'max'])

,max_salary,min_salary,applies,views
mean,8.833622e+04,62352.218073,22.833103,76.776575
median,8.200000e+04,60000.000000,6.000000,25.000000
std,9.068282e+04,59487.692283,54.892826,167.459105
min,1.000000e+01,10.000000,1.000000,1.000000
max,1.300000e+06,800000.000000,1615.000000,5656.000000


#### 3.3 Standardize Data Types
Convert the specified identifier and numeric fields to nullable integer types for consistent handling of missing values.


In [ ]:
# converting the datatypes of the specified columns to 'Int64' for better handling of missing values and consistency
job_postings_df['company_id'] = job_postings_df['company_id'].astype('Int64')
job_postings_df['applies'] = job_postings_df['applies'].astype('Int64')
job_postings_df['remote_allowed'] = job_postings_df['remote_allowed'].astype('Int64')
job_postings_df['views'] = job_postings_df['views'].astype('Int64')

#### 3.4 Validate Company Relationships
Keep only job postings whose company identifiers are present in the cleaned companies dataset.


In [ ]:
# filtering the job_postings_df to keep only the rows with company_id present in companies_df
job_postings_df = job_postings_df[job_postings_df['company_id'].isin(companies_df['company_id'])]
job_postings_df

,job_id,company_id,title,max_salary,min_salary,work_type,location,applies,remote_allowed,views,job_posting_url,application_url,application_type,experience_level,sponsored
1,133114754,77766802,Sales Manager,NaN,NaN,Full-time,"Santa Clarita, CA",<NA>,<NA>,<NA>,https://www.linkedin.com/jobs/view/133114754/?...,NaN,ComplexOnsiteApply,NaN,0
2,133196985,1089558,Model Risk Auditor,NaN,NaN,Contract,"New York, NY",1,<NA>,17,https://www.linkedin.com/jobs/view/133196985/?...,NaN,ComplexOnsiteApply,NaN,0
3,381055942,96654609,Business Manager,NaN,NaN,Full-time,"Forney, TX",<NA>,<NA>,<NA>,https://www.linkedin.com/jobs/view/381055942/?...,NaN,ComplexOnsiteApply,NaN,0
4,529257371,1244539,NY Studio Assistant,NaN,NaN,Full-time,"New York, NY",<NA>,<NA>,2,https://www.linkedin.com/jobs/view/529257371/?...,NaN,ComplexOnsiteApply,NaN,1
5,903408693,3894635,Office Associate,42000.0,37000.0,Full-time,"Albany, GA",5,<NA>,49,https://www.linkedin.com/jobs/view/903408693/?...,NaN,ComplexOnsiteApply,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15881,3701373516,74718032,Sanitation Technician,NaN,NaN,Part-time,"West Columbia, SC",<NA>,<NA>,1,https://www.linkedin.com/jobs/view/3701373516/...,https://aspirebakeriescareers.com/job/West-Col...,OffsiteApply,Entry level,0
15882,3701373522,38897,Unit Secretary,NaN,NaN,Full-time,"Teaneck, NJ",2,<NA>,7,https://www.linkedin.com/jobs/view/3701373522/...,https://recruiting.ultipro.com/HOL1005HNMC/Job...,OffsiteApply,Entry level,0
15883,3701373523,38897,"Radiology Aide, Perdiem",NaN,NaN,Part-time,"Teaneck, NJ",<NA>,<NA>,3,https://www.linkedin.com/jobs/view/3701373523/...,https://recruiting.ultipro.com/HOL1005HNMC/Job...,OffsiteApply,Entry level,0
15884,3701373524,2623,MRI Manager,135000.0,110000.0,Full-time,"New York, NY",<NA>,<NA>,10,https://www.linkedin.com/jobs/view/3701373524/...,https://opportunities.columbia.edu/jobs/mri-ma...,OffsiteApply,Mid-Senior level,0


#### 3.5 Export Job Postings
Save the cleaned job postings dataset for downstream SQL loading and analysis.


In [147]:
job_postings_df.to_csv(os.path.join(parent_dir, "cleaned_datasets", "1_job_postings.csv"), index=False)

### 4. Job Skills & Industries
Combine the job skills and job industries source tables, inspect the merged result, validate job references against the cleaned job postings dataset, and export the final relational dataset.


### Making file#4 (job_skills)

#### 4.1 Inspect Source and Merged Data
Review the source tables and the merged job skills dataset before applying the job-level relational filter.


In [112]:
job_skills_df = pd.merge(df1, df2, on='job_id', how='inner')

In [113]:
df1.info()
df2.info()
job_skills_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27899 entries, 0 to 27898
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   job_id     27899 non-null  int64
 1   skill_abr  27899 non-null  str  
dtypes: int64(1), str(1)
memory usage: 436.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 21993 entries, 0 to 21992
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   job_id       21993 non-null  int64
 1   industry_id  21993 non-null  int64
dtypes: int64(2)
memory usage: 343.8 KB
<class 'pandas.DataFrame'>
RangeIndex: 40088 entries, 0 to 40087
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   job_id       40088 non-null  int64
 1   skill_abr    40088 non-null  str  
 2   industry_id  40088 non-null  int64
dtypes: int64(2), str(1)
memory usage: 939.7 KB


In [114]:
job_skills_df.sample(5)

,job_id,skill_abr,industry_id
33256,3701302062,BD,82
19835,3697360056,MGMT,96
39310,3701369200,MRKT,104
14206,3694100907,IT,96
33314,3701302210,BD,96


In [115]:
job_skills_df[job_skills_df.job_id == 3693049437]

,job_id,skill_abr,industry_id
4300,3693049437,SALE,43
4301,3693049437,BD,43


#### 4.2 Validate Job Relationships
Retain only job skills and industry records whose job identifiers exist in the cleaned job postings dataset.


In [149]:
job_skills_df = job_skills_df[job_skills_df['job_id'].isin(job_postings_df['job_id'])]
job_skills_df

,job_id,skill_abr,industry_id
19,3691797924,ENG,6
20,3691797924,IT,6
54,3692300361,MRKT,44
55,3692300361,SALE,44
56,3692300361,ACCT,44
...,...,...,...
40083,3701373522,ADM,14
40084,3701373523,HCPR,14
40085,3701373524,OTHR,14
40086,3701373527,BD,14


#### 4.3 Export Job Skills & Industries
Save the validated job skills and industry dataset for downstream relational analysis.


In [150]:
job_skills_df.to_csv(os.path.join(parent_dir, "cleaned_datasets", "2_job_skills_industry.csv"), index=False)